# Chapter 9 — Multi-Agent Systems

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

In [1]:
import os, shutil, subprocess, time, urllib.request, urllib.error


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    # Lightning persistent path first, then standard locations
    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh"
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("✓ Using Ollama Cloud")

✓ Using Ollama Cloud


## Examples

### Example 1: Manual Dispatch with UseSubAgentTool

Console logging is enabled below so you can watch the dispatched sub-agent run — its log lines are tagged `[hailstone]`, distinguishing them from the coordinator's own logs in later examples.

In [2]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import ToolCall
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.subagents import SubAgentSpec, UseSubAgentTool
from llm_agents_from_scratch.tools import SimpleFunctionTool

enable_console_logging(logging.INFO)

model = "qwen3.5:397b-cloud" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None
llm = OllamaLLM(host=host, model=model, think=False, json_prompt_mode=use_cloud)


def next_number(x: int) -> int:
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(func=next_number)
hailstone_agent = LLMAgent(llm=llm, tools=[next_number_tool])

spec = SubAgentSpec(
    name="hailstone",
    description="Computes Hailstone sequences using next_number.",
    agent=hailstone_agent,
    max_steps=20,
)
tool = UseSubAgentTool(subagents_registry={spec.name: spec})

tool_call = ToolCall(
    tool_name="from_scratch__use_subagent",
    arguments={
        "name": "hailstone",
        "task": (
            "Compute the full Hailstone sequence for 5 step by step "
            "using next_number, until you reach 1. Report how many "
            "steps it took."
        ),
    },
)
result = await tool(tool_call=tool_call)
print(result.content)

[hailstone] INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the full Hailstone sequence for 5 step by step using next_number, until you reach 1. Report how many steps it took.
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the full Hailstone sequence for 5 step by step using next_number, until you reach 1. Report how many steps it took.
[hailstone] INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: next_number
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: 16
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "c6e9e798-5603-45ca-9444-79688bfb5c35",
    "tool_name": "next_number",
    "argu...[TRUNCATED]
[hailstone] INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Execute the tool call with id 'c6e9e798-5603-45ca-9444-79688bfb5c35' using the 'next_number' tool with argument x=16.
[hailstone] INFO (llm_agents_fs.TaskHandler) :  

### Example 2: Building the Coordinator

In [3]:
from llm_agents_from_scratch import LLMAgentBuilder
from llm_agents_from_scratch.subagents.recipes import explore_subagent, general_subagent

# explore only does lookups, so a small local model is plenty; general
# does open-ended computation, so it shares the coordinator's own model
slm = OllamaLLM(model="qwen3:8b", think=False)

coordinator = await (
    LLMAgentBuilder()
    .with_llm(llm)  # the bigger model from Example 1
    .with_subagents([general_subagent(llm), explore_subagent(slm)])
    .build()
)

print(coordinator.subagents_registry)

{'general': SubAgentSpec(name='general', description='General-purpose subagent for open-ended research, multi-step tasks, or work that benefits from an isolated context. Equipped with file reading and Python execution.', agent=<llm_agents_from_scratch.agent.llm_agent.LLMAgent object at 0x7dcd806920d0>, max_steps=20), 'explore': SubAgentSpec(name='explore', description='Read-only subagent for quickly finding and reading files. Use for lookups and fact-finding, not multi-step work.', agent=<llm_agents_from_scratch.agent.llm_agent.LLMAgent object at 0x7dcd80691e50>, max_steps=10)}


### Example 3: Router/Triage

In [4]:
from llm_agents_from_scratch.data_structures import Task

# a lookup task — should route to explore (reads hailstone_known_sequences.json)
task_lookup = Task(
    instruction=(
        "Look up the Hailstone sequence for 6 in "
        "hailstone_known_sequences.json and report it."
    ),
)
result_lookup = await coordinator.run(task_lookup)
print(result_lookup.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Look up the Hailstone sequence for 6 in hailstone_known_sequences.json and report it.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Look up the Hailstone sequence for 6 in hailstone_known_sequences.json and report it.
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
[explore] INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Find and read the file hailstone_known_sequences.json, then report the Hailstone sequence for the number 6.
[explore] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Find and read the file hailstone_known_sequences.json, then report the Hailstone sequence for the number 6.
[explore] INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__read_file
[explore] INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: {
  "6": {"sequence": [6, 3, 10, 5, 16, 8, 4, 2, 1], "steps": 8},
  "12": {"sequence": [12, 6, 3, 10,

### Example 4: Parallel Fan-Out

In [5]:
task_fanout = Task(
    instruction=(
        "Dispatch three sub-agent calls in the same response so they run "
        "concurrently: use general to compute the Hailstone sequence for "
        "4, use general to compute it for 8, and use explore to look up "
        "the sequence for 12 in hailstone_known_sequences.json. Report "
        "which of the three starting numbers took the most steps."
    ),
)
# gather() runs the three dispatches concurrently, but wall-clock
# speedup isn't guaranteed: a single local Ollama instance queues
# concurrent requests to the *same* model unless OLLAMA_NUM_PARALLEL is
# raised. Cloud models parallelize for free.
result_fanout = await coordinator.run(task_fanout)
print(result_fanout.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Dispatch three sub-agent calls in the same response so they run concurrently: use general to compute the Hailstone sequence for 4, us...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Dispatch three sub-agent calls in the same response so they run concurrently: use general to compute the Hailstone sequence for 4,...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
[explore] INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Look up the Hailstone sequence for starting number 12 in the file hailstone_known_sequences.json. Report the sequence and the number ...[TRUNCATED]
[explore] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Look up the Hailstone sequence for sta